# J5 — Comparer, démontrer et défendre un système NLP

**NLP Avancé (BERT, GPT, Hugging Face) · M2 IA · Jour 5 · 240 min · Chrys NIONGOLO**

Travail en binôme : alterner la personne qui conduit et celle qui explique. Enregistrer une copie dans Drive, puis conserver le notebook exécuté et le JSON final.

## À la fin, vous saurez

- Préenregistrer une comparaison sur la même tâche et le même test.
- Comparer TF-IDF, zero-shot et encodeur adapté avec qualité, coût et erreurs.
- Livrer une démonstration Gradio et une carte de modèle ; publier seulement sur choix explicite.

## Parcours de l’après-midi

| Étape | Minutes |
| Contrat de projet et données | 25 |
| Trois systèmes et préparation | 65 |
| Test, incertitude et analyse | 50 |
| Démonstration et carte de modèle | 45 |
| Soutenances et restitution | 55 |

Les durées sont des budgets d’activité incluant lecture, discussion, essais et débrief ; ce ne sont pas des durées de calcul promises. Les corpus intégrés sont originaux, fictifs et minuscules. Ils servent à comprendre un protocole, pas à certifier une qualité métier.

**Règle d’expérience :** écrire une prédiction avant de lancer une variante ; changer un seul facteur ; choisir sur la validation ; ouvrir le test après ce choix.


## 0. Préparer Colab

Importer ce fichier dans https://colab.research.google.com/ via Fichier → Importer.
Choisir Exécution → Modifier le type d’exécution → Python 3 → GPU T4 si disponible.
La disponibilité du T4 dépend des quotas Colab. Aucun compte Hugging Face n’est requis
pour les modèles publics utilisés ici. Une connexion internet est nécessaire pour les
bibliothèques et les poids ; les données pédagogiques sont déjà dans le notebook.

Exécuter la cellule suivante **avant les autres imports**. Si elle demande un redémarrage,
choisir Exécution → Redémarrer la session puis relancer depuis le début. Ne pas réinstaller
un autre PyTorch dans un runtime CUDA fonctionnel. Les versions sont volontairement fixées.
Aucun entraînement n’est lancé par cette cellule. Aucun envoi au Hub n’est automatique.

**Runpod L4 facultatif :** ce notebook fonctionne dans Jupyter avec Python 3.10+
et PyTorch CUDA préinstallé. Les chemins sont relatifs ; aucun import Google Colab
n’est requis. Arrêter le Pod après sauvegarde des résultats pour interrompre sa facturation.


In [ ]:
PINNED = {'numpy': '2.1.3', 'pandas': '2.2.3', 'matplotlib': '3.10.6', 'scikit-learn': '1.7.2', 'transformers': '4.56.2', 'tokenizers': '0.22.0', 'huggingface-hub': '0.35.3', 'accelerate': '1.10.1', 'datasets': '4.1.1', 'gradio': '5.49.1'}
import sys, subprocess, importlib.metadata as metadata
assert sys.version_info >= (3, 10), "Utiliser Python 3.10 ou ultérieur."
module_names = {"scikit-learn": "sklearn", "sentence-transformers": "sentence_transformers",
                "huggingface-hub": "huggingface_hub", "rouge-score": "rouge_score"}
to_install = []
loaded_mismatch = []
for package, version in PINNED.items():
    module = sys.modules.get(module_names.get(package, package))
    if module is not None and getattr(module, "__version__", version) != version:
        loaded_mismatch.append(package)
    try:
        current = metadata.version(package)
    except metadata.PackageNotFoundError:
        current = None
    if current != version:
        to_install.append(f"{package}=={version}")
if to_install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *to_install])
if loaded_mismatch:
    raise RuntimeError("Redémarrer la session Colab, puis relancer depuis le début : "
                       + ", ".join(loaded_mismatch))
print("Versions de référence installées. Continuer avec la cellule suivante.")


In [ ]:
import os, json, time, random, platform, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
versions = {p: metadata.version(p) for p in PINNED}
versions["python"] = platform.python_version()
results = {"seed": SEED, "versions": versions, "validation": "exécuté par le binôme"}
OUTPUT = Path("resultats")
OUTPUT.mkdir(exist_ok=True)
display(pd.Series(versions, name="version effective"))


In [ ]:
import torch
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
versions["torch"] = torch.__version__
results["device"] = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU"
print("Périphérique :", results["device"])
if DEVICE == "cuda":
    print("VRAM totale (Gio) :", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
else:
    print("Pas de GPU : les sections indiquées restent utilisables ; ne pas présenter un modèle non entraîné comme adapté.")


## 1. Contrat du mini-projet

Parcours fourni : router les demandes de support dans les cinq catégories du J3. Les vingt cas
de défi ci-dessous sont nouveaux par rapport aux J3/J4 ; ils restent fictifs et petits. Ne les lire
qu’une fois les réglages choisis sur la validation. Pour une autre tâche métier, conserver le même
contrat : baseline pertinente, modèle initial, modèle adapté, même jeu de test et critères annoncés.

Le projet ne se gagne pas au score le plus haut : il faut une comparaison honnête, des erreurs
analysées et une démonstration qui expose ses limites. Si le GPU manque, rendre les résultats
disponibles et identifier précisément les systèmes non exécutés.

Répartir les rôles : protocole/données, expériences, démonstration, présentation. Un binôme peut
cumuler les rôles, mais chacun doit expliquer une décision technique à l’oral.


In [ ]:
SUPPORT = {'livraison': [('Le suivi de mon colis ne bouge plus depuis lundi.', 'Mon paquet semble bloqué au centre de tri.'), ('Le livreur indique livré mais ma boîte est vide.', "Je n'ai rien reçu alors que le suivi annonce une remise."), ('Puis-je choisir un autre point relais ?', 'Je voudrais faire déposer ma commande dans un relais différent.'), ("L'adresse sur mon envoi est incomplète.", 'Il manque le numéro de rue pour la livraison.'), ('Le paquet est reparti chez vous avant mon passage.', "Je n'ai pas pu récupérer le colis à temps au relais."), ('Les deux articles voyagent-ils dans le même carton ?', "Je n'ai qu'un numéro de suivi pour plusieurs produits."), ('Le transporteur a reporté son passage sans date.', "Aucune nouvelle tentative de livraison n'est annoncée."), ('Mon colis doit arriver au bureau fermé samedi.', 'La réception de mon entreprise sera absente lors du passage.')], 'facturation': [("J'ai été débité deux fois pour la même commande.", 'Deux paiements identiques apparaissent sur mon relevé.'), ('Où récupérer la facture de mon achat ?', "J'ai besoin d'un justificatif comptable pour la commande."), ("La remise affichée n'a pas été appliquée au paiement.", 'Le total payé ignore le code promotionnel saisi.'), ('Le paiement est refusé alors que ma carte fonctionne ailleurs.', "Votre page n'accepte pas ma carte bancaire."), ('Le nom de mon entreprise manque sur la facture.', 'Pouvez-vous corriger les informations de facturation ?'), ("Une somme reste en attente après l'annulation.", "L'autorisation bancaire semble encore bloquée."), ('Le reçu est dans une devise différente de mon panier.', 'Je ne comprends pas la conversion sur mon justificatif.'), ('La taxe indiquée sur le document paraît incorrecte.', 'Le montant de TVA de ma facture mérite une vérification.')], 'compte': [("J'ai oublié mon mot de passe.", 'Je ne me souviens plus du code pour accéder à mon espace.'), ("Le courriel de réinitialisation n'arrive pas.", "Le lien pour retrouver mon accès n'est pas dans ma boîte mail."), ("Je voudrais changer l'adresse email de mon profil.", 'Mon espace utilise encore mon ancien courriel.'), ('Comment fermer définitivement mon compte ?', 'Je souhaite supprimer mon profil client.'), ('Le code de connexion à deux facteurs est expiré.', 'La double authentification bloque mon accès.'), ('Je ne reconnais pas une connexion récente.', 'Mon historique montre un accès qui ne vient pas de moi.'), ('Je veux récupérer les données de mon profil.', 'Comment obtenir une copie des informations de mon compte ?'), ('Mon adresse mail semble associée à deux profils.', 'Je retrouve deux espaces clients pour la même identité.')], 'retour': [("La taille reçue ne me convient pas, je veux renvoyer l'article.", 'Je souhaite retourner ce vêtement trop petit.'), ('Le produit arrivé cassé doit vous être retourné.', 'Je voudrais organiser le renvoi de cet article abîmé.'), ("Où trouver l'étiquette de retour ?", "Je n'arrive pas à télécharger le bordereau de renvoi."), ('Mon retour a été reçu, où en est son traitement ?', "Avez-vous vérifié le produit que j'ai renvoyé ?"), ("Je préfère échanger l'article plutôt que le garder.", 'Est-il possible de remplacer ce produit par une autre taille ?'), ("J'ai jeté l'emballage, puis-je tout de même renvoyer ?", "Le carton d'origine est perdu pour mon retour."), ("Je souhaite renvoyer seulement une partie de l'achat.", "Comment retourner un seul article d'une commande multiple ?"), ('Le transporteur refuse mon paquet de retour trop volumineux.', 'Le renvoi ne rentre pas dans le format accepté au relais.')], 'technique': [("L'application se ferme quand j'ouvre mon panier.", 'Votre appli plante au moment de voir les articles ajoutés.'), ('La page produit reste blanche sur mon navigateur.', 'Les fiches articles ne se chargent plus sur le site.'), ('Le bouton de recherche ne fait rien.', 'Je saisis un mot mais la recherche du site ne démarre pas.'), ("L'image du produit ne s'affiche pas.", 'Toutes les photos sont remplacées par une icône cassée.'), ('La version mobile masque le bouton continuer.', "Sur mon téléphone je ne peux pas passer à l'étape suivante."), ("Le site boucle lorsqu'on accepte les cookies.", 'La fenêtre de consentement réapparaît sans cesse.'), ('Votre formulaire refuse les caractères accentués.', "Le champ de saisie signale une erreur quand j'écris é."), ("La page est inutilisable avec mon lecteur d'écran.", "La navigation au clavier n'atteint pas les boutons du site.")]}
LABELS = sorted(SUPPORT)
label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}
rows = []
for label, groups in SUPPORT.items():
    for g, pair in enumerate(groups):
        for variant, text in enumerate(pair):
            rows.append({"text": text, "label": label2id[label], "category": label,
                         "group": f"{label}_{g}", "scenario": g,
                         "split": "train" if g < 5 else "validation" if g == 5 else "test"})
frame = pd.DataFrame(rows)
assert frame.text.is_unique
train_df = frame[frame.split == "train"].reset_index(drop=True)
val_df = frame[frame.split == "validation"].reset_index(drop=True)
test_df = frame[frame.split == "test"].reset_index(drop=True)
for a, b in [(train_df, val_df), (train_df, test_df), (val_df, test_df)]:
    assert set(a.group).isdisjoint(b.group), "Fuite de paraphrases entre partitions."
display(pd.crosstab(frame.category, frame.split))
results["data"] = {"source": "Corpus original fictif CYBERSUP, 2026-10-04",
                   "rows": len(frame), "split_unit": "scénario/paraphrases",
                   "sha256": hashlib.sha256(frame.to_json().encode()).hexdigest()}


In [ ]:
project_contract = {
    "objectif": "Router des tickets fictifs de support parmi cinq intentions",
    "primary_metric": "macro-F1 sur un test commun de 20 nouveaux scénarios",
    "secondary_metrics": ["accuracy", "sorties invalides", "latence moyenne"],
    "selection": "Validation J3 uniquement ; test J5 après gel des réglages",
    "out_of_scope": "Traitement réel de commandes, garantie commerciale, sécurité universelle",
    "hypothese": "À compléter avant les expériences",
    "critere_decision": "À compléter : qualité minimale utile, coût et besoin humain",
}
display(pd.Series(project_contract))


In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, classification_report, ConfusionMatrixDisplay
baseline = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), strip_accents="unicode"),
                         LogisticRegression(max_iter=1000, C=2.0, random_state=SEED))
baseline.fit(train_df.text, train_df.label)
val_pred_baseline = baseline.predict(val_df.text)
baseline_val_f1 = f1_score(val_df.label, val_pred_baseline, labels=list(id2label), average="macro", zero_division=0)
print("Macro-F1 validation baseline :", round(baseline_val_f1, 3))
results["baseline_validation_macro_f1"] = float(baseline_val_f1)


## 2. Réutiliser l’encodeur adapté du J3

Déposer modele_classification.zip dans le panneau Fichiers si vous l’avez conservé.
Sans cet export, le notebook refait le petit entraînement J3 sur GPU. Le modèle importé doit avoir
les mêmes labels et une provenance vérifiée. Un fichier de poids externe non documenté ne constitue
pas une preuve d’adaptation sur les bonnes données.


In [ ]:
MODEL_ID = 'distilbert/distilbert-base-multilingual-cased'
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, TrainingArguments, Trainer, set_seed)
set_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
MAX_LENGTH = 128
def make_dataset(table):
    ds = Dataset.from_dict({"text": table.text.tolist(), "labels": table.label.tolist()})
    return ds.map(lambda batch: tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH),
                  batched=True, remove_columns=["text"])
train_ds, val_ds = make_dataset(train_df), make_dataset(val_df)
token_lengths = [len(tokenizer(text)["input_ids"]) for text in train_df.text]
print("Longueur max avant troncature :", max(token_lengths), "; limite choisie :", MAX_LENGTH)
assert max(token_lengths) <= MAX_LENGTH, "Réexaminer la troncature avant d'entraîner."
def compute_metrics(prediction):
    pred = np.argmax(prediction.predictions, axis=-1)
    return {"accuracy": accuracy_score(prediction.label_ids, pred),
            "macro_f1": f1_score(prediction.label_ids, pred, labels=list(id2label), average="macro", zero_division=0)}
RUN_TRAINING = DEVICE == "cuda"  # Sur CPU, activer volontairement après accord sur le temps disponible.
EPOCHS = 3
LEARNING_RATE = 3e-5
trainer = None
trained_model = None


In [ ]:
# Facultatif : déposer modele_classification.zip (export du J3) dans le panneau Fichiers.
import zipfile
model_dir = Path("modele_classification")
archive = Path("modele_classification.zip")
if archive.exists():
    model_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            dest = (model_dir / member.filename).resolve()
            if not dest.is_relative_to(model_dir.resolve()):
                raise ValueError("Chemin non valide dans le fichier ZIP.")
        z.extractall(model_dir)
if (model_dir / "config.json").exists():
    trained_model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(DEVICE)
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    assert trained_model.config.label2id == label2id, "Le modèle importé n'utilise pas ces labels."
    RUN_TRAINING = False
    print("Modèle J3 importé : vérifier sa provenance et l'absence de cas J5 dans l'entraînement.")
else:
    print("Pas de modèle J3 importé : reproduction du petit entraînement sur GPU si disponible.")


In [ ]:
if RUN_TRAINING:
    set_seed(SEED)
    trained_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID, num_labels=len(LABELS), id2label=id2label, label2id=label2id)
    # La tête de classification neuve est aléatoire : l'avertissement de poids initialisés est attendu.
    args = TrainingArguments(
        output_dir="checkpoints_classification", learning_rate=LEARNING_RATE,
        per_device_train_batch_size=8, per_device_eval_batch_size=8,
        num_train_epochs=EPOCHS, weight_decay=0.01,
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
        fp16=DEVICE == "cuda", bf16=False, report_to="none", seed=SEED,
        data_seed=SEED, logging_steps=5, dataloader_num_workers=0,
        push_to_hub=False, save_safetensors=True)
    trainer = Trainer(model=trained_model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                      processing_class=tokenizer, data_collator=DataCollatorWithPadding(tokenizer),
                      compute_metrics=compute_metrics)
    start = time.perf_counter()
    trainer.train()
    results["classification_train_seconds"] = time.perf_counter() - start
    results["classification_validation"] = trainer.evaluate()
    results["classification_config"] = {"epochs": EPOCHS, "learning_rate": LEARNING_RATE,
                                         "max_length": MAX_LENGTH, "model": MODEL_ID}
    print(results["classification_validation"])
elif trained_model is None:
    print("Entraînement non exécuté. Baseline et analyse du protocole restent disponibles.")
def predict_finetuned(texts):
    if trained_model is None:
        return None
    trained_model.eval()
    output = []
    for start in range(0, len(texts), 8):
        batch = tokenizer(list(texts[start:start + 8]), padding=True, truncation=True,
                          max_length=MAX_LENGTH, return_tensors="pt").to(DEVICE)
        with torch.inference_mode():
            logits = trained_model(**batch).logits
        output.extend(logits.argmax(-1).cpu().tolist())
    return np.array(output)


## 3. Une baseline zero-shot sur exactement la même tâche

Le LLM ne voit aucun exemple annoté dans le prompt : il reçoit seulement les cinq étiquettes
et une instruction de classement. Nous conservons sa réponse brute ; toute sortie non conforme
devient une prédiction invalide et compte comme erreur. Nous ne lui attribuons pas le bon label
à l’aide d’une recherche de mot opportuniste.

Pour borner la mémoire, l’encodeur adapté passe sur CPU pendant les générations. Le modèle 0.5B
est un choix pédagogique de coût : sa qualité zero-shot en français doit être constatée.


In [ ]:
ZERO_SHOT_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
from transformers import AutoModelForCausalLM
RUN_ZERO_SHOT = DEVICE == "cuda"
ztokenizer = None
zmodel = None
ZERO_PROMPT = "Classe le message client dans exactement une catégorie de cette liste : " + ", ".join(LABELS) + ". Réponds uniquement par le nom de la catégorie, sans explication. Message client : "
if trained_model is not None: trained_model.to("cpu")
if DEVICE == "cuda": torch.cuda.empty_cache()
if RUN_ZERO_SHOT:
    ztokenizer = AutoTokenizer.from_pretrained(ZERO_SHOT_MODEL)
    zmodel = AutoModelForCausalLM.from_pretrained(ZERO_SHOT_MODEL, torch_dtype=DTYPE).to(DEVICE).eval()
def predict_zero_shot(texts):
    raw, pred = [], []
    if zmodel is None: return None, None
    for text in texts:
        messages = [{"role": "user", "content": ZERO_PROMPT + text}]
        inputs = ztokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
            return_tensors="pt", return_dict=True).to(DEVICE)
        with torch.inference_mode():
            out = zmodel.generate(**inputs, max_new_tokens=12, do_sample=False, pad_token_id=ztokenizer.eos_token_id)
        answer = ztokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().lower()
        # Tolérer seulement la ponctuation finale et les guillemets de présentation.
        canonical = answer.strip(" .!?:;\"'")
        raw.append(answer)
        pred.append(label2id.get(canonical, -1))
    return np.array(pred), raw
if zmodel is not None:
    validation_zs, raw_validation_zs = predict_zero_shot(val_df.text.tolist())
    print("Zero-shot macro-F1 validation :", f1_score(val_df.label, validation_zs, labels=list(id2label), average="macro", zero_division=0))
    display(pd.DataFrame({"texte": val_df.text, "sortie_brute": raw_validation_zs}))


## 4. Geler les choix, ouvrir le défi

Figer le prompt et les hyperparamètres. Chaque méthode reçoit les mêmes textes dans le même ordre.
La latence ci-dessous est une moyenne de ce run, téléchargements exclus, avec du code et des tailles
de lots différents ; elle n’est pas un benchmark d’infrastructure. La comparaison de qualité porte
sur vingt cas : le bootstrap ci-dessous illustre l’incertitude, sans compenser ce faible effectif.


In [ ]:
CHALLENGE = [('livraison', "Le chauffeur m'a appelé pendant une réunion et a emporté le paquet ; comment reprogrammer la remise ?"), ('livraison', "Le code pour ouvrir la consigne à colis ne m'a pas été envoyé."), ('livraison', "Le suivi situe mon envoi dans une ville où je n'habite pas."), ('livraison', "Le carton devait être livré contre signature mais personne ne s'est présenté."), ('facturation', 'Le prix du panier était de 30 euros et le prélèvement est de 35 euros.'), ('facturation', 'Mon service comptable réclame un document avec le détail hors taxes.'), ('facturation', "Je vois un règlement alors que la validation de l'achat a échoué."), ('facturation', 'Le numéro de facture figurant sur le reçu ne correspond pas à ma commande.'), ('compte', 'Je ne possède plus le téléphone qui reçoit mes codes de connexion.'), ('compte', 'Un message annonce que mon identifiant est désactivé.'), ('compte', 'Comment déconnecter mon profil des anciens appareils ?'), ('compte', "Je reçois des alertes de changement de courriel que je n'ai pas demandé."), ('retour', 'Je souhaite restituer ce cadeau qui ne convient pas à la personne.'), ('retour', 'Le bordereau de renvoi expire avant que je puisse déposer le produit.'), ('retour', "J'ai envoyé le mauvais article dans mon retour, comment régulariser ?"), ('retour', "L'échange demandé doit porter sur une autre couleur et pas une autre taille."), ('technique', "Le menu de votre site ne s'ouvre pas dans Firefox."), ('technique', 'Le téléchargement du catalogue affiche une erreur serveur.'), ('technique', "Les boutons de la page se superposent quand j'agrandis le texte."), ('technique', 'La recherche efface mon texte à chaque caractère saisi.')]
challenge = pd.DataFrame(CHALLENGE, columns=["category", "text"])
challenge["label"] = challenge.category.map(label2id)
challenge["group"] = [f"j5_{i:02d}" for i in range(len(challenge))]
assert set(challenge.text).isdisjoint(frame.text)
assert challenge.text.is_unique
test_predictions = {}
times = {}
t0 = time.perf_counter()
test_predictions["tfidf_logreg"] = baseline.predict(challenge.text)
times["tfidf_logreg"] = time.perf_counter() - t0
t0 = time.perf_counter()
zs_pred, zs_raw = predict_zero_shot(challenge.text.tolist())
if zs_pred is not None:
    test_predictions["zero_shot"] = zs_pred
    times["zero_shot"] = time.perf_counter() - t0
import gc
if zmodel is not None:
    del zmodel
    zmodel = None
    gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()
if trained_model is not None:
    trained_model.to(DEVICE)
    if DEVICE == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    test_predictions["encodeur_finetune"] = predict_finetuned(challenge.text.tolist())
    if DEVICE == "cuda": torch.cuda.synchronize()
    times["encodeur_finetune"] = time.perf_counter() - t0
metric_table = []
y = challenge.label.to_numpy()
for method, pred in test_predictions.items():
    metric_table.append({"methode": method,
        "macro_f1": f1_score(y, pred, labels=list(id2label), average="macro", zero_division=0),
        "accuracy": accuracy_score(y, pred), "invalides": int(np.sum(pred == -1)),
        "ms_par_texte": 1000 * times[method] / len(y), "n": len(y)})
display(pd.DataFrame(metric_table))
errors = challenge.copy()
for method, pred in test_predictions.items():
    errors[method] = [id2label.get(int(p), "INVALIDE") for p in pred]
if zs_raw is not None: errors["zero_shot_brut"] = zs_raw
display(errors)


In [ ]:
def paired_bootstrap_f1(y, pred_a, pred_b, repeats=1000):
    # Un scénario par ligne dans ce défi ; garder le groupe comme unité sur un autre corpus.
    rng = np.random.default_rng(SEED)
    deltas = []
    for _ in range(repeats):
        idx = rng.integers(0, len(y), size=len(y))
        a = f1_score(y[idx], pred_a[idx], labels=list(id2label), average="macro", zero_division=0)
        b = f1_score(y[idx], pred_b[idx], labels=list(id2label), average="macro", zero_division=0)
        deltas.append(a - b)
    return {"delta_moyen": float(np.mean(deltas)), "intervalle_95_percentile": np.quantile(deltas, [.025, .975]).tolist()}
bootstrap = None
if "encodeur_finetune" in test_predictions:
    bootstrap = paired_bootstrap_f1(y, test_predictions["encodeur_finetune"], test_predictions["tfidf_logreg"])
    print("Différence encodeur adapté - baseline :", bootstrap)
error_analysis = [{"cas": "À compléter", "cause": "À compléter", "action": "À compléter"} for _ in range(5)]
decision = "À compléter : système retenu, compromis, abstention humaine, limites du test."


## 5. Démontrer un comportement utile

La démonstration propose soit TF-IDF soit le modèle adapté effectivement disponible. Elle affiche
une catégorie et un rappel de périmètre ; elle ne prétend pas prendre une décision commerciale.
Ne pas présenter un score softmax comme une confiance calibrée. Ajouter un cas vide et un texte très long.

Gradio n’est pas lancé automatiquement. LAUNCH_DEMO=True ouvre un serveur dans le runtime,
avec share=False. Selon le contexte Colab, l’affichage local peut être indisponible ; le test direct
de la fonction reste utilisable. Créer un tunnel public demande un choix distinct et ne fait pas partie
du parcours par défaut. Ne saisir aucune donnée sensible dans une démo partagée.


In [ ]:
import gradio as gr
choices = ["tfidf_logreg"] + (["encodeur_finetune"] if trained_model is not None else [])
def classify_demo(text, method):
    text = text.strip()
    if not text: return "Saisir un message fictif pour tester la démonstration."
    if len(text) > 2000: return "Texte trop long pour cette démonstration : limiter à 2 000 caractères."
    if method == "encodeur_finetune":
        prediction = predict_finetuned([text])
        if prediction is None: return "Modèle adapté indisponible dans cette session."
        label = id2label[int(prediction[0])]
    else:
        label = id2label[int(baseline.predict([text])[0])]
    return f"Catégorie proposée : {label}.\nPrototype pédagogique : une personne valide les cas ambigus. Aucun dossier réel n'est consulté."
print(classify_demo("Je n'arrive pas à retrouver la facture de mon achat", choices[0]))
assert "Saisir" in classify_demo(" ", choices[0])
assert "trop long" in classify_demo("a" * 2001, choices[0])
demo = gr.Interface(fn=classify_demo,
    inputs=[gr.Textbox(lines=4, label="Message client fictif"), gr.Dropdown(choices, value=choices[0], label="Méthode")],
    outputs=gr.Textbox(label="Résultat"), title="CYBERSUP — routage de support",
    description="Démonstration M2 IA. Données fictives uniquement. Les catégories ne couvrent pas toutes les demandes.",
    examples=[["J'ai été prélevé deux fois", choices[0]], ["Le menu ne s'ouvre plus", choices[0]]],
    flagging_mode="never")
LAUNCH_DEMO = False
if LAUNCH_DEMO:
    demo.launch(share=False, inline=True, prevent_thread_lock=True)
else:
    print("Démonstration testée directement ; serveur non lancé. Activer LAUNCH_DEMO si souhaité.")


## 6. Carte de modèle et publication optionnelle privée

Documenter données, langue, séparation, labels, métriques obtenues, limites et usages interdits.
L’export ci-dessous ne contient ni token d’accès, ni données de compte, ni notebook avec des secrets.
Relire le dossier avant l’upload. Le Hub propose un dépôt de modèle, pas automatiquement une API de production.

PUBLISH_PRIVATE reste False. Pour publier volontairement : renseigner votre propre identifiant,
passer le booléen à True puis saisir un token dans le champ masqué. Le code crée un dépôt privé et
refuse l’upload si un dépôt existant portant ce nom est public. Le token n’est jamais écrit dans le notebook.


In [ ]:
publication_dir = Path("publication_modele")
if trained_model is not None:
    publication_dir.mkdir(exist_ok=True)
    trained_model.save_pretrained(publication_dir, safe_serialization=True)
    tokenizer.save_pretrained(publication_dir)
    model_card = (
        "---\nlanguage: fr\nlicense: apache-2.0\nlibrary_name: transformers\npipeline_tag: text-classification\n---\n"
        "# Prototype pédagogique de routage CYBERSUP\n\n"
        "Modèle de base : " + MODEL_ID + ". Corpus original fictif français de 80 textes. "
        "Séparation par scénario ; comparaison J5 sur 20 cas fictifs nouveaux.\n\n"
        "## Résultats de cette exécution\n\n" + json.dumps(metric_table, ensure_ascii=False, indent=2) +
        "\n\n## Limites\n\nMicro-corpus synthétique, ambiguïtés entre catégories, aucun test métier réel. "
        "Ne pas utiliser pour traiter automatiquement des remboursements, données personnelles ou décisions sensibles. "
        "Aucune confiance calibrée ni disponibilité de service garantie.\n\n"
        "## Décision du groupe\n\n" + decision + "\n")
    (publication_dir / "README.md").write_text(model_card, encoding="utf-8")
    (publication_dir / "protocole.json").write_text(json.dumps({"contract": project_contract,
        "data": results["data"], "versions": versions, "metrics": metric_table}, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Fichiers à relire :", sorted(p.name for p in publication_dir.iterdir()))
PUBLISH_PRIVATE = False
REPO_ID = "votre-identifiant/cybersup-nlp-projet"
if PUBLISH_PRIVATE:
    if trained_model is None: raise ValueError("Aucun modèle adapté à publier.")
    if REPO_ID.startswith("votre-identifiant/"): raise ValueError("Renseigner votre propre identifiant Hugging Face.")
    from getpass import getpass
    from huggingface_hub import HfApi
    token = getpass("Token Hugging Face écriture (saisie masquée) : ")
    try:
        api = HfApi(token=token)
        api.create_repo(repo_id=REPO_ID, repo_type="model", private=True, exist_ok=True)
        if not api.model_info(REPO_ID).private:
            raise ValueError("Le dépôt existant est public : choisir un autre nom privé avant l'upload.")
        api.upload_folder(repo_id=REPO_ID, repo_type="model", folder_path=publication_dir,
            allow_patterns=["*.json", "*.safetensors", "*.txt", "*.model", "*.md"],
            commit_message="Prototype pédagogique NLP et protocole")
        print("Dépôt privé : https://huggingface.co/" + REPO_ID)
    finally:
        token = None
        if "api" in globals(): del api
else:
    print("Aucune publication effectuée.")


## 7. Soutenance : cinq minutes, puis questions

Présenter la tâche et le coût d’une erreur (45 s), le protocole (60 s), la comparaison et deux erreurs
(90 s), la démonstration (45 s), la décision et la prochaine expérience (60 s).

Remettre notebook exécuté, JSON, cinq erreurs commentées, carte de modèle et capture ou démonstration.
La publication Hub est facultative et ne remplace aucun de ces éléments. Chaque membre explique
pourquoi son test est séparé et quelle preuve manquerait avant un usage réel.


In [ ]:
results.update({"contract": project_contract, "metrics": metric_table, "bootstrap": bootstrap,
                "predictions": errors.to_dict(orient="records"), "error_analysis": error_analysis,
                "decision": decision, "zero_shot_model": ZERO_SHOT_MODEL,
                "zero_shot_prompt": ZERO_PROMPT, "available_methods": list(test_predictions)})
conclusion_binome = "À compléter : système retenu, preuves, limites et prochaine expérience."


## Ressources pour prolonger

- [HF — Démonstrations](https://huggingface.co/learn/llm-course/fr/chapter9/1)
- [Gradio — interfaces](https://www.gradio.app/guides/quickstart)
- [HF Hub — dépôts privés et publication](https://huggingface.co/docs/huggingface_hub/en/guides/repository)
- [HF — Model cards](https://huggingface.co/docs/hub/model-cards)

Liens et API consultés le 4 octobre 2026. Les exercices et données de ce notebook sont originaux. Les téléchargements de modèles et l’exécution Colab doivent être vérifiés avant la séance.


## Trace à rendre

Compléter vos réponses et ajouter une observation appuyée par une sortie réelle.
Exécuter la dernière cellule, télécharger le JSON dans le panneau Fichiers de Colab,
puis télécharger votre notebook avec ses sorties. Les fichiers du runtime sont temporaires.
Ne pas remplacer une mesure absente par un chiffre plausible. Indiquer les étapes non exécutées.


In [ ]:
results["notebook"] = '07_j5_projet'
results["conclusion_binome"] = globals().get("conclusion_binome", "À compléter")

result_file = OUTPUT / "j5_resultats.json"
result_file.write_text(json.dumps(results, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
print("Fichier à télécharger :", result_file.resolve())
